In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

In [2]:
DATA_DIR = Path.cwd().parent / 'datasets' / 'processed'
RAW_DIR = Path.cwd().parent / 'datasets' / 'raw'
MODELS_DIR = Path.cwd().parent / 'models'

In [3]:
# Load the comparison table saved from the evaluation notebook
results_df = pd.read_csv(DATA_DIR / 'model_evaluation_results.csv' )
results_df = results_df.sort_values(by='Test_RMSE', ascending=True)

best_model_name = results_df.iloc[0]['Model']
print(f"Best model: {best_model_name}")

# Load the corresponding saved model file
best_model = joblib.load(MODELS_DIR / f'{best_model_name}_model.pkl')

Best model: Cat_Boost_Regressor


In [4]:
train_data = pd.read_csv(DATA_DIR / 'final_train.csv')
X_train = train_data.drop('SalePrice', axis=1)
y_train = train_data['SalePrice']

X_test_final = pd.read_csv(DATA_DIR / 'final_test.csv')

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test_final.shape}")

Train shape: (1458, 232)
Test shape: (1459, 232)


In [5]:
best_model.fit(X_train, y_train)

predictions_log = best_model.predict(X_test_final)

In [6]:
# SalePrice was log1p-transformed during EDA, so reverse it to get real dollar values
predictions_real = np.expm1(predictions_log)

print(predictions_real[:5])

[124202.85612173 165901.35352155 184452.43248035 200841.52957415
 186430.40976903]


In [7]:
sample_submission = pd.read_csv(RAW_DIR / 'sample_submission.csv')

assert len(predictions_real) == len(sample_submission), "Row count mismatch!"

In [8]:
submission = pd.DataFrame({ 'Id': sample_submission['Id'],
                             'SalePrice': predictions_real  
                          })

submission.to_csv(DATA_DIR / 'submission.csv', index=False)
print("submission.csv saved")
submission.head()

submission.csv saved


,Id,SalePrice
0,1461,124202.856122
1,1462,165901.353522
2,1463,184452.432480
3,1464,200841.529574
4,1465,186430.409769
